In [ ]:
import streamlit as st
import joblib
import pandas as pd
import numpy as np
import json
import plotly.express as px
import plotly.graph_objects as go

# 페이지 설정
st.set_page_config(
    page_title = "자동차 연비 예측 앱",
    page_icon = "🚗", # 이모지 단축키 : win + .
    layout="wide"
)

# 제목 
st.title("🚗자동차 연비 예측 애플리케이션")
st.markdown("---")

# 모델 로드(캐싱)
@st.cache_resource
def load_model():
    try:
        return joblib.load("./day04/mpg_model.joblib")
    except Exception as e:
        st.error("모델 파일을 찾을 수 없습니다")
        st.error(f"e")
        return None

# 모델 정보 로드
def load_model_info():
    try:
        with open('./day04/model_info.json', 'r', encoding='utf-8') as f:
            return json.load(f)
    except:
        return None

# 모델 및 정보 로드
model = load_model()
model_info = load_model_info()

if model is None: # 모델 로드 실패시
    st.stop()

# 사이드 바
st.sidebar.header("⚙️ 설정")

# 모델 정보 표시
st.sidebar.subheader("📊 모델 선능")

if model_info:
    st.sidebar.metric("결정계수", f"{model_info['r2_score'] : .3f}")
    st.sidebar.metric("RMSE", f"{model_info['rmse'] : .3f}")
    st.sidebar.metric("MSE", f"{model_info['mse'] : .3f}")

# 탭 영역 나누기
tab1, tab2, tab3 = st.tabs(["🔮예측", "📈모델정보", "📊데이터 분석"])

# 탭1 : 예측
with tab1:
    st.header("연비 예측")

    col1, col2 = st.columns([2, 1])

    with col1:
        # 자동차 무게 입력폼
        st.subheader("입력 정보")
        weight = st.number_input(
            "자동차 무게 (lbs)",
            min_value=1000,
            max_value=6000,
            value=3000,
            step=100,
            help="예측할 자동차의 무게를 입력하세요"
        )

        # 예측 버튼
        if st.button("🔮 연비 예측하기", type="primary", use_container_width=True):
            # 예측 실행
            input_data = np.array([[weight]])
            predicted_mpg = model.predict(input_data)[0]
            # st.write(predicted_mpg)

            # 세션 상태에 저장
            if 'predictions' not in st.session_state:
                st.session_state.predictions = []
            st.session_state.predictions.append({
                "weight" : weight,
                "predicted_mpg" : predicted_mpg,
            })

    with col2:
        # 예측 결과 표시
        if st.session_state.get("predictions"):
            latest = st.session_state.predictions[-1]
            st.subheader("예측 결과")
            st.metric(
                label="예측 연비",
                value=f"{latest['predicted_mpg'] : .2f}"
            )
            st.caption("단위 : mpg(miles per gallon)")

            # 결과 해석
            with st.expander("📖 결과 해석"):
                st.write(f"**입력값** : {latest['weight']} lbs")
                st.write(f"**예측 연비** : {latest['predicted_mpg'] : .2f} mpg")
                st.write(f"""
                    **해석** :
                    - 무게가 {latest['weight']} lbs인 자동차의 예상 연비는 {latest['predicted_mpg'] : .2f} mpg 입니다.
                    - 이 값은 학습 데이터의 패턴을 기반으로 계산되었습니다.
                """)

                if model_info:
                    st.write(f"""
                        **오차 범위**
                        - 모델의 RMSE {model_info['rmse'] : .2f} mpg 입니다.
                        - 실제 연비는 예측값에서 평균적으로 {model_info['rmse'] : .2f} mpg 정도 차이가 날 수 있습니다
                    """)

# 탭2 : 모델 정보(성능 지표, 회귀식, 성능 지표 해석)
with tab2:
    st.header("모델 정보")

    if model_info:
        col1, col2 = st.columns(2)

        with col1:
            # 성능 지표
            st.subheader("📊 성능지표")
            st.metric("결정계수", f"{model_info['r2_score'] : .3f}")
            st.metric("평균 제곱 오차(MSE)", f"{model_info['mse'] : .3f}")
            st.metric("루트 평균 제곱 오차(RMSE)", f"{model_info['rmse'] : .3f}")

        with col2:
            # 회귀식 : 독립변수와 종속변수 간의 관계 확인
            st.subheader("📐 회귀식")
            # 수식 표현 : st.latex()
            # mpg = 기울기 x weight + 절편
            st.latex(f"mpg = {model_info['coef'] : .3f} \\times weight + {model_info['intercept'] : .3f}")

            st.write("** 계수 해석 **")
            st.write(f"- 기울기 : {model_info['coef'] : .3f}")
            st.write(f"- 무게가 1 lbs 증가할 때마다 연비가 평균적으로 {abs(model_info['coef']) : .3f} mpg 감소")
            st.write("- 절편 : {model_info['intercept'] : .3f}")

    # 성능 지표 해석
    st.subheader("📖 성능 지표 해석")
    with st.expander("자세한 해석 보기"):
        st.write(f"""
        **1. 결정계수 = {model_info['r2_score'] : .3f}**
       - 모델이 연비 변동의 약 {model_info['r2_score'] * 100 : .3f}%를 설명합니다.
       - {model_info['r2_score'] * 100 : .3f}%는 무게로 설명 가능하고, 나머지는 다른 요인에 의해 설명됩니다.

       **2. RMSE = {model_info['rmse'] : .2f}mpg**
       - 예측값이 실제값과 평균적으로 {model_info['rmse'] : .2f}mpg 정도 차이가 납니다.
       - 이 값이 작을수록 모델의 예측 정확도가 높습니다.

       **3. 모델의 한계**
       - 무게 외에도 엔진 크기, 공기역학적 특성 등이 연비에 영향을 미칩니다.
       - 더 정확한 예측을 위해서는 추가 변수가 필요할 수 있습니다.
        """)

# 탭3 : 데이터 분석(시각화)
with tab3:
    st.header("데이터 분석")

    # 데이터 생성
    import seaborn as sns
    df = sns.load_dataset('mpg')
    df =df.dropna() # 결측치 제거

    # 시각화
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("무게 vs 연비 산점도")
        fig_scatter = px.scatter(
            df,
            x = 'weight',
            y = 'mpg',
            title = '무게와 연비의 관계',
            labels={'weight' : '무게(lbs)', 'mpg' : '연비(mpg)'}
            # trendline='ols' # 회귀선 추가
        )
        st.plotly_chart(fig_scatter, use_container_width=True)
    with col2:
        st.subheader("예측 히스토리")
        if st.session_state.get('predictions'):
            pred_df = pd.DataFrame(st.session_state.predictions)
            st.dataframe(pred_df)
        else:
            st.info("예측을 수행하면 히스토리가 표시됩니다.")
    # 통계 정보
    st.subheader("데이터 통계")
    st.dataframe(df[['weight', 'mpg']].describe())